<center><font size=20> <bold> AI Business Research Agent </font size> </bold></center>


## **Objective**



Build an AI-powered research agent that can find, verify, organize, and
summarize business information from publicly available internet sources.
The agent should behave like a professional researcher rather than a simple web
scraper.


## **Installing Packages**

The necessary packages are installed.

*  google-generativeai -- lets talk to gemini model
*  ddgs -- access to DuckDuckGo search
*  pandas -- for business records into tables
*  requests -- lets python to visit webpage and download raw contents
*  beautifulsoup4 - convert raw html to readable
*  rapidfuzz - for dedeuplication

In [1]:
!pip install -q google-generativeai
!pip install -q ddgs
!pip install -q pandas
!pip install -q requests
!pip install -q beautifulsoup4
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import requests
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

## **Gemini Setup**

Gemini model has been connected succesfully which is used at the end of the pipeline to generate a professional research summary report from the collected business data.

In [3]:
from google.colab import userdata
import google.generativeai as genai
api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini conneced successfully!")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini conneced successfully!


## **Utility Functions**

### **Helper Functions 1: Extract Domain from URL**

Extracts the domain name from a given URL. This standardizes website references and enables consistent source classification and business identification across different web pages.

In [4]:
from urllib.parse import urlparse
def extract_domain(url):
  try:
    domain = urlparse(url).netloc
    domain = domain.replace("www.", "")
    return domain
  except:
    return ""

In [5]:
print(extract_domain("https://www.healthpartners.com"))

healthpartners.com


### **Helper Functions 2 : Domain to Business Name**

Converts a website domain into a human readable business name. This help generate business records when company names are not explicitly available from the search results.

In [6]:
def domain_to_business_name(domain):
  parts = domain.split(".")
  if len(parts) >=2:
    name = parts[-2]
  else:
    name = parts[0]
  return(name.replace("-"," ").title())

In [7]:
print(domain_to_business_name("mayoclinic.org"))
print(domain_to_business_name("healthpartners.com"))
print(domain_to_business_name("providers.mhealthfairview.org"))

Mayoclinic
Healthpartners
Mhealthfairview


### **Helper Functions 3 : Web Page Text Collection**

Retrieves and extracts visible text contect from a webpage. The collected text is later used for contact information, business verification, and information analysis.

In [8]:
def collect_page_text(url):
  try:
    response = requests.get(url, headers = {"User-Agent": "Mozilla/5.0"}, timeout=10)
    if response.status_code !=200:
      return None
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text[:10000]
  except Exception:
    return None

### **Helper Function 4 : Email Extraction**

Uses regular expressions to scan webpage text and identify email addresses. Handles both standard formats like .com and international domains like .in, .co.uk. Duplicate emails are automatically removed.

In [9]:
def extract_emails(text):
    if not text:
        return []
    # Already handles international domains (.in, .uk, .co.in etc.)
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,10}\b'
    emails = re.findall(email_pattern, text)
    emails = [email.lower().strip() for email in emails]
    return list(set(emails))

### **Helper Function 5 : Phone Number Extraction**

Identifies and extracts phone numbers from webpage content using pattern matching. Extracted numbers are validated and standardized before being included in the final business profile.

In [10]:
def extract_phone_numbers(text):
  if not text:
    return[]
  phone_pattern = r'''
        (?:
            \+?[\d\s\-\(\)]{7,20}   # international format with optional +
            (?:ext|x|ext\.)\s*\d+   # optional extension
        |
            \+?[\d\s\-\(\)]{7,20}   # standard international
        )
    '''

  phones = re.findall(phone_pattern, text, flags=re.VERBOSE)
  cleaned = []
  for phone in phones:
    digits = re.sub(r'\D', '', phone)
    if 7 <= len(digits) <=15:
      cleaned.append(phone.strip())
  return list(set(cleaned))

In [11]:
def clean_phone_numbers(phones):
    valid_phones = []

    for phone in phones:
        digits = re.sub(r"\D", "", phone)

        if 7 <= len(digits) <= 15:
            valid_phones.append(phone.strip())
    return list(set(valid_phones))

    return valid_phones

### **Helper Function 6 : Address Extraction**

Detects and extracts physical addresses from webpage text using multiple regex patterns. Helps enrich business profiles with location information.

In [12]:
def extract_addresses(text):
    if not text:
        return []

    # Pattern 1: US addresses (existing)
    us_pattern = (
        r'\d+\s+[A-Za-z0-9\s,.-]+'
        r'(?:Ave|Avenue|St|Street|Rd|Road|Blvd|Boulevard|Dr|Drive|Ln|Lane|Way|Ct|Court)'
        r'[A-Za-z0-9\s,.-]*\d{5}'
    )

    # Pattern 2: UK addresses (postcode like SW1A 1AA)
    uk_pattern = (
        r'\d+\s+[A-Za-z0-9\s,.-]+'
        r'[A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2}'
    )

    # Pattern 3: Indian addresses (PIN code like 600001)
    india_pattern = (
        r'\d+[,\s]+[A-Za-z0-9\s,.-]+'
        r'(?:Street|Road|Nagar|Colony|Layout|Main|Cross|Avenue)'
        r'[A-Za-z0-9\s,.-]*\d{6}'
    )

    # Pattern 4: Generic — any line with a number and city-like words
    generic_pattern = (
        r'\d+[,\s]+[A-Za-z\s]{5,50}[,\s]+'
        r'[A-Za-z\s]{3,30}[,\s]+\d{4,6}'
    )

    addresses = []
    for pattern in [us_pattern, uk_pattern, india_pattern, generic_pattern]:
        found = re.findall(pattern, text, flags=re.IGNORECASE)
        addresses.extend([a.strip() for a in found])

    return list(set(addresses))

### **Helper Function 7 : Extract Business from Directory**

Visits HealthGrades, Yelp, and Topnpi directory pages and extracts individual business or doctor names directly from the page content. Only processes valid business listing pages - city homepages and navigation pages are automatically skipped to avoid junk results.

In [13]:
def extract_businesses_from_directory_page(url, source_name):
    """
    Visits a Yelp or Healthgrades list page and
    extracts individual business names from it.
    """
    page_text = collect_page_text(url)
    if not page_text:
        return []

    businesses = []

    if "yelp.com" in url:
        valid_yelp_patterns = ["/biz", "/search?", "find_desc="]
        if not any (pattern in url for pattern in valid_yelp_patterns):
          return []

        pattern = r'\b([A-Z][a-z]+(?: [A-Z][a-z]+){1,5})\b'
        candidates = re.findall(pattern, page_text)
        skip_words = [
            "Read More", "Get Directions", "Write Review",
            "Phone Number", "Business Hours", "United States",
            "More Info", "Log In", "Sign Up", "Privacy Policy", "Terms of Service", "Cookie Policy", "Elite Squad"
        ]
        for name in candidates:
            if name not in skip_words and len(name) > 5:
                businesses.append({
                    "business_name": name,
                    "domain": "yelp.com",
                    "website": url,
                    "record_source": source_name
                })

    elif "healthgrades.com" in url:
        pattern = r'\b(Dr\.?\s[A-Z][a-z]+(?:\s[A-Z][a-z]+){1,3}|[A-Z][a-z]+(?: [A-Z][a-z]+){1,4}(?:Clinic|Center|Hospital|Health|Medical|Neurology|Cardiology|Associates|Group))\b'
        candidates = re.findall(pattern, page_text)
        for name in candidates:
            businesses.append({
                "business_name": name,
                "domain": "healthgrades.com",
                "website": url,
                "record_source": source_name
            })

    elif "topnpi.com" in url:
        pattern = r'(?:Dr\.\s)?([A-Z][a-z]+(?:\s[A-Z][a-z]+){1,3}),\s(?:MD|DM|DO|PHD|NP|PA)'
        candidates = re.findall(pattern, page_text)
        for name in candidates:
          businesses.append({
              "business_name": "Dr." + name,
              "domain": "topnpi.com",
              "website": url,
              "record_source": source_name
          })
    return businesses

## **Query Agent**

Defining a function which takes user's input query and spilts it into two parts - the business type and the location


In [14]:
def understand_query(user_query):
  pattern = r"\s(?:in|near to|near|around|close to)\s+"
  parts = re.split(pattern, user_query, flags=re.IGNORECASE)

  if len(parts)>=2:
    return {
      "business_type": parts[0].strip(),
      "location": parts[1].strip()
    }
  else:
    return {
      "business_type": user_query.strip(),
      "location": ""
    }

In [15]:
import re
print(understand_query("Dentists in Austin"))
print(understand_query("Dentists near Austin"))
print(understand_query("Dentists near to Austin"))
print(understand_query("Dentists around Austin"))
print(understand_query("Dentists close to Austin"))

{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}


## **Search Query Generator**

Generates multiple search queries to gather business information from diverse online source.

In [16]:
def generate_search_queries(business_type, location):
  return[
      f"{business_type} in {location}",
      f"{business_type} in {location} official website",
      f"{business_type} in {location} reviews",
      f"{business_type} in {location} linkedin",
      f"{business_type} in {location} directory",
      f"{business_type} {location} yelp",
      f"{business_type} {location} healthgrades",
      f"{business_type} {location} site:topnpi.com"
  ]

In [17]:
result = understand_query("Neurologists in Minnesota")
queries = generate_search_queries(result["business_type"], result["location"])
for query in queries:
  print(query)

Neurologists in Minnesota
Neurologists in Minnesota official website
Neurologists in Minnesota reviews
Neurologists in Minnesota linkedin
Neurologists in Minnesota directory
Neurologists Minnesota yelp
Neurologists Minnesota healthgrades
Neurologists Minnesota site:topnpi.com


## **Search Agent**

Performs web searches using the generated queries and gathers relevant business information, including the titles, URLs and descriptions, into a structured dataframe for subsequent processing.

In [38]:
from ddgs import DDGS

def search_business(queries, max_results=5):
  all_results = []
  with DDGS() as ddgs:
    for query in queries:
      try:
        results = ddgs.text(query, max_results=max_results)
        if results: # Only process if there are results
            for result in results:
              all_results.append({
                "query": query,
                "title": result.get("title", ""),
                "href": result.get("href", ""),
                "body": result.get("body", "")
            })
      except Exception as e:
        print(f"Search error:{e}")

  # Ensure the DataFrame always has the expected columns, even if empty
  if not all_results:
      return pd.DataFrame(columns=["query", "title", "href", "body"])
  return pd.DataFrame(all_results)

In [19]:
result = understand_query("Neurologists in Minnesota")
queries = generate_search_queries(result["business_type"],result["location"])
all_results_df = search_business(queries, max_results=5)
all_results_df.head()

,query,title,href,body
0,Neurologists in Minnesota,The Best Neurologists in Minnesota | US News,https://health.usnews.com/doctors/neurologists...,We found 696 neurologists in Minnesota. The av...
1,Neurologists in Minnesota,Best Neurologists in Minnesota (2026) | Top-Ra...,https://doctor.webmd.com/providers/specialty/n...,Find Top Neurologists in Minnesota. Drill down...
2,Neurologists in Minnesota,"Best Neurologists in Minneapolis, MN (2026) - ...",https://doctor.webmd.com/providers/specialty/n...,"Discover top Neurologists in Minneapolis, MN -..."
3,Neurologists in Minnesota,25 of the Best Neurologists Near Me in Minneso...,https://www.medifind.com/specialty/neurology/U...,Looking for the best neurologist or nerve spec...
4,Neurologists in Minnesota,Top 10 Neurologists in Minnesota | America Top 10,https://americatop10.com/minnesota/top10/neuro...,The 6 best neurologists in Minnesota across 2 ...


## **Source Classfication Agent**

Analyzes the domain of each search result and categorizes it as an Official Website, Directory, Government Source, Search Engine, or Social Media platform. This classification helps assess source reliability and prioritize authoritative sources for business intelligence and profile generation.

In [20]:
def classify_source(domain):
  domain = domain.lower()
  if any(
      site in domain
      for site in [
          "linkedin",
          "facebook",
          "youtube",
          "instagram",
          "tiktok",
          "x.com",
          "twitter"
      ]
  ):
   return "Social Media"

  if any(
      site in domain
      for site in [
          "google",
          "bing",
          "duckduckgo"
      ]
  ):
    return "Search Engine"

  if any(
      site in domain
      for site in [
          "healthgrades",
          "usnews",
          "webmd",
          "medifind",
          "vitals",
          "ratemds",
          "yelp",
          "yellowpages",
          "castleconnolly",
          "threebestrated"
      ]
 ):
   return "Directory"

  if ".gov" in domain:
    return "Government"

  return "Official Website"

In [21]:
test_domains = [
    "health.usnews.com",
    "medifind.com",
    "healthpartners.com",
    "google.com",
    "facebook.com",
    "linkedin.com"
]
for domain in test_domains:
  print(f"{domain} -> {classify_source(domain)}")

health.usnews.com -> Directory
medifind.com -> Directory
healthpartners.com -> Official Website
google.com -> Search Engine
facebook.com -> Social Media
linkedin.com -> Social Media


## **Research Sources Agent**

Enriches search results by extracting website domains and applying source classification. The agent transforms raw search results into a structured research dataset containing source metadata, enabling reliable source evaluation and downstream business profile generation.

In [40]:
def build_research_sources_df(all_results_df):
  research_sources_df = all_results_df.copy()
  # Ensure 'href' column exists, fill with None if missing
  if 'href' not in research_sources_df.columns:
      research_sources_df['href'] = None
  research_sources_df["domain"] = (research_sources_df["href"].apply(extract_domain))
  research_sources_df["source_type"] = (research_sources_df["domain"].apply(classify_source))
  return research_sources_df

In [23]:
research_sources_df = build_research_sources_df(all_results_df)
research_sources_df[
    ["title",
    "domain",
    "source_type",
    ]
].head(10)

,title,domain,source_type
0,The Best Neurologists in Minnesota | US News,health.usnews.com,Directory
1,Best Neurologists in Minnesota (2026) | Top-Ra...,doctor.webmd.com,Directory
2,"Best Neurologists in Minneapolis, MN (2026) - ...",doctor.webmd.com,Directory
3,25 of the Best Neurologists Near Me in Minneso...,medifind.com,Directory
4,Top 10 Neurologists in Minnesota | America Top 10,americatop10.com,Official Website
5,The Best Neurologists in Minnesota - Health Us...,health.usnews.com,Directory
6,"Best Neurologists in Minneapolis, MN (2026) | ...",doctor.webmd.com,Directory
7,20 Best Neurologists In Minnesota | Healthgrades,healthgrades.com,Directory
8,Best Neurologists in Minnesota (2026) | Top-Ra...,doctor.webmd.com,Directory
9,Our neurologists | HealthPartners & Park Nicollet,healthpartners.com,Official Website


## **Business Discovery Agent**

Identifies real businesses from the research sources using two approches. Official Website sources use the domain name to generate a business name. Directory sources use the page title to extract the business name. Both are combined, irrelevant entries are filtered out, and individual businesses are scraped directly through HealthGrades, and Yelp directory pages to maximise the number of businesses found

In [24]:
from types import NoneType
def build_business_records_df(research_sources_df):

  exclude_keywords = ["veterinary", "directory", "directories","welli", "medicalnewstoday", "mentaltherapy", "news",
                      "blog", "americatop10", "danielaragon", "find a", "search", "best neurologists", "top neurologists",
                      "how to", "topnpi", "princeton", "scribd"]

  ui_words = ["privacy", "policy", "login", "sign up", "terms", "service", "loading", "categories", "restaurants", "copyright",
              "careers", "investors", "advertise", "support", "mobile", "developers"
              ]

  official_df = research_sources_df[research_sources_df["source_type"]=="Official Website"].copy()
  official_df["business_name"] = official_df["domain"].apply(domain_to_business_name)
  official_df["record_source"] = "Official Website"

  directory_df = research_sources_df[research_sources_df["source_type"]=="Directory"].copy()

  def extract_name_from_title(title):
    skip_phrases = [
    "best 10", "best 15", "best 20", "top 10", "top 65",
    "top neurologist", "top doctors", "find a", "search results",
    "near me", "neurologists near", "doctors near",
    "in minneapolis", "in birmingham", "in dallas",
    "in austin", "in chicago", "in houston",
    "doctors who", "physicians who", "3 best", "5 best", "10 best", "15 best", "20 best",
    "best electricians", "best plumbers", "best lawyers", "top electricians", "top plumbers"
]
    title_lower = title.lower()
    if any(phrase in title_lower for phrase in skip_phrases):
      return None
    for sep in ["|", "-", "–", "—", "·"]:
      if sep in title:
        name = title.split(sep)[0].strip()
        if len(name) > 3:
          return name
    return title.strip()

  directory_df["business_name"] = directory_df["title"].apply(extract_name_from_title)
  directory_df["record_source"] = "Directory"

  combined_df = pd.concat([official_df, directory_df], ignore_index=True)

  combined_df = combined_df[~combined_df["business_name"].str.lower().str.contains("|".join(exclude_keywords), na=False)]

  combined_df = combined_df[~combined_df["business_name"].str.lower().str.contains("|".join(ui_words), na=False)]

  combined_df = combined_df[["business_name", "domain", "href", "record_source"]].rename(columns={"href": "website"})

  combined_df = combined_df[combined_df["business_name"].str.len() > 3]
  combined_df = combined_df.reset_index(drop=True)

  combined_df = combined_df[combined_df["business_name"].notna()]
  combined_df = combined_df[combined_df["business_name"].str.len() > 3]
  combined_df = combined_df.reset_index(drop=True)

  directory_urls = research_sources_df[research_sources_df["domain"].str.contains("yelp|healthgrades|topnpi", na=False)]["href"].tolist()

  extra_businesses = []
  for url in directory_urls[:3]:
    if "yelp.com" in url:
      extra_businesses.extend(extract_businesses_from_directory_page(url, "Directory"))
    elif "healthgrades.com" in url:
      extra_businesses.extend(extract_businesses_from_directory_page(url, "Directory"))

  if extra_businesses:
        extra_df = pd.DataFrame(extra_businesses)
        combined_df = pd.concat([combined_df, extra_df], ignore_index=True)
        combined_df = combined_df.reset_index(drop=True)

  return combined_df

In [25]:
business_records_df = build_business_records_df(research_sources_df)
print("Total Businesses found:", len(business_records_df))
print("\nBy source type:")
print(business_records_df["record_source"].value_counts())
business_records_df[["business_name", "domain", "record_source"]].head(15)

Total Businesses found: 71

By source type:
record_source
Directory           67
Official Website     4
Name: count, dtype: int64


,business_name,domain,record_source
0,Healthpartners,healthpartners.com,Official Website
1,Healthline,care.healthline.com,Official Website
2,Healthpartners,healthpartners.com,Official Website
3,Healthpartners,healthpartners.com,Official Website
4,UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC,yelp.com,Directory
5,Dr. Micah Yost,healthgrades.com,Directory
6,Dr. Joon Uhm,healthgrades.com,Directory
7,Dr. Ilo Leppik,healthgrades.com,Directory
8,Dr. Mithri Junna,healthgrades.com,Directory
9,Dr. Orhun Kantarci,healthgrades.com,Directory


## **Business Deduplication Agent**

Uses RapidFuzz similarity matching to identify duplicate and remove duplicate business records. Returns the cleaned list along with the count of how many duplicates were removed.

In [26]:
from rapidfuzz import fuzz
def deduplicate_business_df(business_records_df, similarity_threshold=90):
  deduplicated=[]
  duplicates_removed = 0

  for _, row in business_records_df.iterrows():
    business_name = row["business_name"]
    is_duplicate = False

    for existing in deduplicated:
      score = fuzz.token_sort_ratio(business_name.lower(), existing["business_name"].lower())
      if score >= similarity_threshold:
        is_duplicate = True
        duplicates_removed += 1
        break

    if not is_duplicate:
      deduplicated.append(row.to_dict())

  result_df = pd.DataFrame(deduplicated, columns=business_records_df.columns)
  return result_df, duplicates_removed

In [27]:
deduplicated_business_df, duplicates_removed = deduplicate_business_df(business_records_df)
print("Before deduplication:", len(business_records_df))
print("After deduplication:", len(deduplicated_business_df))

deduplicated_business_df

Before deduplication: 71
After deduplication: 23


,business_name,domain,website,record_source
0,Healthpartners,healthpartners.com,https://www.healthpartners.com/care/find/docto...,Official Website
1,Healthline,care.healthline.com,https://care.healthline.com/find-care/specialt...,Official Website
2,UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC,yelp.com,https://www.yelp.com/biz/university-of-minneso...,Directory
3,Dr. Micah Yost,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
4,Dr. Joon Uhm,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
5,Dr. Ilo Leppik,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
6,Dr. Mithri Junna,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
7,Dr. Orhun Kantarci,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
8,Dr. Steven Sabers,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
9,Dr. Joshua Kramer,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory


## **Business Verfication Agent**

Assigns a confidence score to each verified business using the type of source and the number of supporting sources. Official websites are considered highly reliable, while businesses appearing across multiple sources receive increased confidence.

In [28]:
def build_verified_business_df(deduplicated_business_df, business_records_df):
  source_counts = (business_records_df.groupby("business_name")["domain"]
  .nunique()
  .reset_index(name="source_count")
  )

  verification_df = deduplicated_business_df.merge(source_counts, on="business_name", how="left")

  def assign_confidence(row):
    if row["source_count"] >= 3 or row["record_source"] == "Official Website":
      return "High"
    elif row["source_count"] >= 2 or row["record_source"] == "Directory":
      return "Medium"
    else:
      return "Low"

  verification_df["confidence"] = verification_df.apply(assign_confidence, axis=1)
  return verification_df

In [29]:
verification_df = build_verified_business_df(deduplicated_business_df, business_records_df)
print("Total verified businesses:", len(verification_df))
print("\nConfidence breakdown:")
print(verification_df["confidence"].value_counts())
verification_df.head(10)

Total verified businesses: 23

Confidence breakdown:
confidence
Medium    21
High       2
Name: count, dtype: int64


,business_name,domain,website,record_source,source_count,confidence
0,Healthpartners,healthpartners.com,https://www.healthpartners.com/care/find/docto...,Official Website,1,High
1,Healthline,care.healthline.com,https://care.healthline.com/find-care/specialt...,Official Website,1,High
2,UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC,yelp.com,https://www.yelp.com/biz/university-of-minneso...,Directory,1,Medium
3,Dr. Micah Yost,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
4,Dr. Joon Uhm,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
5,Dr. Ilo Leppik,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
6,Dr. Mithri Junna,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
7,Dr. Orhun Kantarci,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
8,Dr. Steven Sabers,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
9,Dr. Joshua Kramer,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium


## **Contact Information Agent**

Visits verified business websites and extracts contact information including addresses, phone numbers, and email addresses. The agent enriches business records with publicly available contact details to support business outreach and profile generation.

In [44]:
def build_contact_df(verification_df):
  contact_records = []
  page_cache = {}
  for idx, row in verification_df.iterrows():
    try:
      website = row["website"]
      if website in page_cache:
        page_text = page_cache[website]
      else:
        page_text = collect_page_text(website)
      page_cache[website] = page_text

      if page_text:
        phones = extract_phone_numbers(page_text)
        phones = clean_phone_numbers(phones)
        emails = extract_emails(page_text)
        addresses = extract_addresses(page_text)
      else:
        phones = []
        emails = []
        addresses = []

      contact_records.append({
          "business_name": row["business_name"],
          "address": addresses[0] if addresses else None,
          "phones": phones,
          "emails": emails,
          "source_url": website
      })
      print(f"{idx}: {row['business_name']}")

    except Exception as e:
      print(f"{idx}: {row['business_name']}, - ERROR: {e}")
      contact_records.append({
          "business_name": row["business_name"],
          "address": None,
          "phones": [],
          "emails": [],
          "source_url": row.get("website", "")
      })
  # Ensure the DataFrame always has the expected columns, even if empty
  if not contact_records:
      return pd.DataFrame(columns=["business_name", "address", "phones", "emails", "source_url"])
  return pd.DataFrame(contact_records)

In [31]:
contact_df = build_contact_df(verification_df)
print("Business processed:", len(contact_df))
print("Business with phones:", contact_df["phones"].apply(len).gt(0).sum())
print("Business with emails:", contact_df["emails"].apply(len).gt(0).sum())
print("Business with addresses:", contact_df["address"].notna().sum())
contact_df.head(10)

0: Healthpartners
1: Healthline
2: UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC
3: Dr. Micah Yost
4: Dr. Joon Uhm
5: Dr. Ilo Leppik
6: Dr. Mithri Junna
7: Dr. Orhun Kantarci
8: Dr. Steven Sabers
9: Dr. Joshua Kramer
10: Dr. Kenneth Hoj
11: Dr. Diego Carvalho
12: Dr. Dimitrios Giannakidis
13: Dr. Carrie Robertson
14: Dr. Yumna Saeed
15: Dr. Fred Lux
16: Dr. Syed Shahkhan
17: Dr. Rupert Exconde
18: Dr. Nadeem Iqbal
19: Dr. Bryan Klassen
20: Dr. Matthew Roller
21: Dr. Oliver Ni
22: Dr. Sotirios Parashos
Business processed: 23
Business with phones: 21
Business with emails: 0
Business with addresses: 21


,business_name,address,phones,emails,source_url
0,Healthpartners,"8170 33rd Ave S, Bloomington, MN 55425",[],[],https://www.healthpartners.com/care/find/docto...
1,Healthline,None,"[51-60 (47) 61-70 (51, 40 (67) 41-50 (97)]",[],https://care.healthline.com/find-care/specialt...
2,UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC,None,[],[],https://www.yelp.com/biz/university-of-minneso...
3,Dr. Micah Yost,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
4,Dr. Joon Uhm,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
5,Dr. Ilo Leppik,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
6,Dr. Mithri Junna,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
7,Dr. Orhun Kantarci,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
8,Dr. Steven Sabers,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...
9,Dr. Joshua Kramer,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...


## **Final Output Agent**

Combines verified business records and extracted contact information into a unified business intelligence dataset. The agent consolidates business details, contact information, source evidence, and confidence scores into a structured final output for analysis and decision-making.

In [32]:
def build_final_output_df(verification_df, contact_df):
  final_output_df = (verification_df.merge(contact_df, on="business_name", how="left"))
  return final_output_df[
      [
          "business_name",
          "address",
          "phones",
          "emails",
          "website",
          "source_count",
          "confidence",
          "record_source",
          "source_url"
      ]
  ]

In [33]:
final_output_df = build_final_output_df(verification_df, contact_df)
print("Total Businesses:", len(final_output_df))
print("High Confidence Businesses:", (final_output_df["confidence"]=="High").sum())
print("Medium Confidence Businesses:", (final_output_df["confidence"]=="Medium").sum())
print("\nColumns:", final_output_df.columns.tolist())
final_output_df

Total Businesses: 23
High Confidence Businesses: 2
Medium Confidence Businesses: 21

Columns: ['business_name', 'address', 'phones', 'emails', 'website', 'source_count', 'confidence', 'record_source', 'source_url']


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Healthpartners,"8170 33rd Ave S, Bloomington, MN 55425",[],[],https://www.healthpartners.com/care/find/docto...,1,High,Official Website,https://www.healthpartners.com/care/find/docto...
1,Healthline,None,"[51-60 (47) 61-70 (51, 40 (67) 41-50 (97)]",[],https://care.healthline.com/find-care/specialt...,1,High,Official Website,https://care.healthline.com/find-care/specialt...
2,UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC,None,[],[],https://www.yelp.com/biz/university-of-minneso...,1,Medium,Directory,https://www.yelp.com/biz/university-of-minneso...
3,Dr. Micah Yost,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
4,Dr. Joon Uhm,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
5,Dr. Ilo Leppik,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
6,Dr. Mithri Junna,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
7,Dr. Orhun Kantarci,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
8,Dr. Steven Sabers,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
9,Dr. Joshua Kramer,2 more provider attributes 3601 Minnesota Dr S...,"[55812 (218) 206-634, 15700 37, (612) 473-0657]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...


## **Research Summary Agent**

Sends the final business dataset to Gemini with a structured prompt, which generates a professional research report in Markdown format. The report includes a Search Summary, Executive Summary, Key Businesses Identified, Research Source Overview, Data Quality Summary, Important Findings, and Conclusion.

In [34]:
def generate_research_summary(user_query, final_output_df, research_sources_df, duplicates_removed=0):
  business_details = final_output_df.to_dict("records")
  source_types = (research_sources_df["source_type"]
                  .value_counts()
                  .to_dict()
                  )
  total = len(final_output_df)
  with_phone = final_output_df["phones"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
  with_address = final_output_df["address"].notna().sum()
  with_email = final_output_df["emails"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()

  prompt = f""" You are a senior business research analyst.

  User Query:{user_query}

  Search statistics:
  -- Businesses Found : {total + duplicates_removed}
  -- Businesses Verified : {total}
  -- Duplicate Records Removed: {duplicates_removed}
  -- Sources Searched : {len(research_sources_df)}

  Data Quality:
  -- Records with Phone : {round(with_phone/total*100)}%
  -- Records with Address : {round(with_address/total*100)}%
  -- Records with Email : {round(with_email/total*100)}%

  Business Details: {business_details}

  Research Source Distribution: {source_types}

  Generate a professional business research report in Markdown format.

  Use Exactly the following structure:

  # Business Research Report

  ## Search Summary
  -- Query : {user_query}
  -- Businesses Found : {total + duplicates_removed}
  -- Businesses Verified : {total}
  -- Duplicate Records Removed: {duplicates_removed}
  -- Sources Searched : {len(research_sources_df)}

  ## Executive Summary
  provide a concise overview of the market and research findings.

  ## Key Business Identified
  For each business include:
  -- Business Name
  -- Website
  -- Source Count
  -- Confidence Level
  -- Address (if available)
  -- Phone Numbers (if available)

Important:
-- Report Source Count exactly as provided in the dataset.
-- Report Confidence Level exactly as provided in the dataset.
-- Do not modify, reinterpret, estimate, or recalculate confidence values.
-- Do NOT generate business descriptions.
-- Only report information explicitly present in the provided dataset.
-- If a field is unavailable, write "Not Available",


  ## Research Source Overview
  Include:
  -- Total sources analyzed
  -- Breakdown of source types
  -- observations about source reliability

  ## Data Quality Summary
  -- Records with Phone Number : {round(with_phone/total*100)}%
  -- Records with Address : {round(with_address/total*100)}%
  -- Records with Email : {round(with_email/total*100)}%

## Important Findings
Provide 3-5 bullet point insights.

## Conclusion
Provide a brief concluding paragraph.

Formatting Requirements:
-- Use proper Markdown headings(#, ##).
-- Use bullet points where appropriate.
-- Keep the report professional and concise.
-- Do not invent facts that are not supported by the provided data.


  """

  response = model.generate_content(prompt)
  return response.text

In [35]:
research_summary = generate_research_summary(
    user_query="Neurologists in Minnesota",
    final_output_df=final_output_df,
    research_sources_df=research_sources_df,
    duplicates_removed=duplicates_removed
)
from IPython.display import Markdown, display
display(Markdown(research_summary))

# Business Research Report

## Search Summary
-- Query : Neurologists in Minnesota
-- Businesses Found : 71
-- Businesses Verified : 23
-- Duplicate Records Removed: 48
-- Sources Searched : 40

## Executive Summary
This report provides a concise overview of neurologists identified in Minnesota. The research process initially found 71 businesses, which were refined to 23 verified unique entities after removing 48 duplicate records. While contact phone numbers and physical addresses are well-covered, with 91% availability for both, a significant data gap exists as no email addresses were found for any of the verified businesses. The identified businesses include both clinics and individual practitioners, primarily sourced from directories and official websites.

## Key Business Identified

-- Business Name: Healthpartners
-- Website: https://www.healthpartners.com/care/find/doctors/neurologists/
-- Source Count: 1
-- Confidence Level: High
-- Address: 8170 33rd Ave S, Bloomington, MN 55425
-- Phone Numbers: Not Available

-- Business Name: Healthline
-- Website: https://care.healthline.com/find-care/specialty/neurology/mn/minneapolis
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: 51-60 (47) 61-70 (51, 40 (67) 41-50 (97)

-- Business Name: UNIVERSITY OF MINNESOTA HEALTH NEUROLOGY CLINIC
-- Website: https://www.yelp.com/biz/university-of-minnesota-health-neurology-clinic-minneapolis-3
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Micah Yost
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Joon Uhm
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Ilo Leppik
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Mithri Junna
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Orhun Kantarci
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Steven Sabers
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Joshua Kramer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Kenneth Hoj
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Diego Carvalho
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Dimitrios Giannakidis
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Carrie Robertson
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Yumna Saeed
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Fred Lux
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Syed Shahkhan
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Rupert Exconde
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Nadeem Iqbal
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Bryan Klassen
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Matthew Roller
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Oliver Ni
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

-- Business Name: Dr. Sotirios Parashos
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 2 more provider attributes 3601 Minnesota Dr Ste 200 Edina, MN 55435
-- Phone Numbers: 55812 (218) 206-634, 15700 37, (612) 473-0657

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    -- Directory: 27
    -- Official Website: 11
    -- Social Media: 2
-- Observations about source reliability: The majority of identified records came from Directory sources, which typically provided a "Medium" confidence level. Official Websites contributed a smaller number of records but with a "High" confidence level. Social Media sources were minimally used.

## Data Quality Summary
-- Records with Phone Number : 91%
-- Records with Address : 91%
-- Records with Email : 0%

## Important Findings
*   A substantial number of initial records (71) were found, but a high duplicate rate resulted in 23 verified, unique businesses, indicating effective deduplication.
*   The data set exhibits strong coverage for critical contact points, with 91% of verified records including both a phone number and a physical address.
*   A notable deficiency is the complete absence of email addresses (0% coverage) across all verified business records, limiting direct digital outreach.
*   Many of the verified entities are individual neurologists listed under a common address, predominantly sourced from directories like Healthgrades, suggesting a focus on individual practitioners.
*   Confidence levels vary, with records from official websites being "High" confidence, while directory-sourced records are primarily "Medium" confidence.

## Conclusion
The research successfully identified and verified 23 neurologists and clinics in Minnesota, providing robust data for phone numbers and addresses. The primary area for improvement lies in acquiring email contact information, which is currently entirely missing. Future research efforts should prioritize enriching the dataset with email addresses to enhance its utility for direct communication and targeted outreach initiatives.

## **Master Agent**

Orchestrates the entire search pipeline with a single function call. Takes any user query, runs it through all agents in sequence - from query understanding to report generation - and returns all intermediate and final outputs. Supports any business type and location, including international queries.

In [36]:
def run_agent(user_query):
  query_info = understand_query(user_query)
  business_type = query_info["business_type"]
  location = query_info["location"]

  queries = generate_search_queries(business_type, location)
  all_results_df = search_business(queries, max_results=5)
  research_sources_df = build_research_sources_df(all_results_df)
  business_records_df = build_business_records_df(research_sources_df)

  deduplicated_business_df, duplicates_removed = deduplicate_business_df(business_records_df)

  verification_df = build_verified_business_df(deduplicated_business_df, business_records_df)
  contact_df = build_contact_df(verification_df)
  final_output_df = build_final_output_df(verification_df, contact_df)

  research_summary = generate_research_summary(
      user_query=user_query,
      final_output_df=final_output_df,
      research_sources_df=research_sources_df,
      duplicates_removed = duplicates_removed
  )

  return {
      "research_sources_df": research_sources_df,
      "business_records_df": business_records_df,
      "deduplicated_business_df": deduplicated_business_df,
      "duplicates_removed": duplicates_removed,
      "verification_df": verification_df,
      "contact_df": contact_df,
      "final_output_df": final_output_df,
      "research_summary": research_summary
  }

In [39]:
results = run_agent("Neurologists in Minnesota")
results["final_output_df"]

0: Noranclinic
1: Minneapolisclinic
2: Ourhealthnetwork
3: Dr. Hans Pinkert
4: Dr. Ilo Leppik
5: Dr. Saugat Dey
6: Dr. Kenneth Hoj
7: Dr. Dimitrios Giannakidis
8: Dr. Rupert Exconde
9: Dr. Zara Fatima
10: Dr. Rwoof Reshi
11: Dr. Yumna Saeed
12: Dr. Sotirios Parashos
13: Dr Golden Valley
14: Dr. Thomas Schriefer
15: Dr. Fred Lux
16: Dr. Syed Shahkhan
17: Dr. Rammohan Sankaraneni
18: Dr. Oliver Ni
19: Dr. William Schmalstieg
20: Dr. Darwin Ramirez Abreu
21: Dr. Micah Yost
22: Dr. Joon Uhm
23: Dr. Mithri Junna
24: Dr. Orhun Kantarci
25: Dr. Steven Sabers
26: Dr. Joshua Kramer
27: Dr. Diego Carvalho
28: Dr. Carrie Robertson
29: Dr. Nadeem Iqbal
30: Dr. Bryan Klassen
31: Dr. Matthew Roller


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Noranclinic,None,"[(612) 879-0722, (612) 879-1000]",[],https://www.noranclinic.com/,1,High,Official Website,https://www.noranclinic.com/
1,Minneapolisclinic,None,[],[],https://minneapolisclinic.com/,1,High,Official Website,https://minneapolisclinic.com/
2,Ourhealthnetwork,None,[],[],https://ourhealthnetwork.com/neurologist/mn,1,High,Official Website,https://ourhealthnetwork.com/neurologist/mn
3,Dr. Hans Pinkert,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
4,Dr. Ilo Leppik,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
5,Dr. Saugat Dey,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
6,Dr. Kenneth Hoj,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
7,Dr. Dimitrios Giannakidis,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
8,Dr. Rupert Exconde,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
9,Dr. Zara Fatima,None,[],[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...


In [41]:
from IPython.display import Markdown, display
display(Markdown(results["research_summary"]))

# Business Research Report

## Search Summary
-- Query : Neurologists in Minnesota
-- Businesses Found : 45
-- Businesses Verified : 32
-- Duplicate Records Removed: 13
-- Sources Searched : 40

## Executive Summary
The search for "Neurologists in Minnesota" successfully identified 32 verified businesses or practitioners from an initial pool of 45 records, after removing 13 duplicates. While a solid count of entities was found, the overall data quality is very low, particularly concerning contact information. Only 3% of records include a phone number, and no records have addresses or email addresses, which significantly limits the utility of this dataset for direct outreach or detailed market analysis. The majority of identified entities appear to be individual practitioners listed in directories, rather than comprehensive clinic profiles.

## Key Business Identified
-- Business Name: Noranclinic
-- Website: https://www.noranclinic.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: (612) 879-0722, (612) 879-1000

-- Business Name: Minneapolisclinic
-- Website: https://minneapolisclinic.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Ourhealthnetwork
-- Website: https://ourhealthnetwork.com/neurologist/mn
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Hans Pinkert
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Ilo Leppik
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Saugat Dey
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Kenneth Hoj
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Dimitrios Giannakidis
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Rupert Exconde
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Zara Fatima
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Rwoof Reshi
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Yumna Saeed
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Sotirios Parashos
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr Golden Valley
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Thomas Schriefer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Fred Lux
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Syed Shahkhan
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Rammohan Sankaraneni
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Oliver Ni
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. William Schmalstieg
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Darwin Ramirez Abreu
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Micah Yost
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Joon Uhm
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Mithri Junna
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Orhun Kantarci
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Steven Sabers
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Joshua Kramer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Diego Carvalho
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Carrie Robertson
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Nadeem Iqbal
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Bryan Klassen
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Matthew Roller
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    -- Directory: 30
    -- Official Website: 8
    -- Social Media: 2
-- Observations about source reliability: The vast majority of records (30 out of 40 sources) were extracted from directories. While these sources identified numerous practitioners, they generally yielded "Medium" confidence levels and lacked comprehensive contact details such as addresses, phone numbers, and emails. In contrast, official websites (8 sources) provided "High" confidence levels and, when available, more complete contact information, but were a smaller proportion of the total sources. Social media accounted for a minor portion of the sources.

## Data Quality Summary
-- Records with Phone Number : 3%
-- Records with Address : 0%
-- Records with Email : 0%

## Important Findings
*   A total of 32 unique and verified neurologists or neurological clinics were identified in Minnesota, indicating a substantial presence of practitioners.
*   The most significant data quality issue is the severe lack of essential contact information, with 0% of records containing an address or email, and only 3% including a phone number.
*   The majority of the identified entities are individual neurologists listed within health directories (e.g., Healthgrades), rather than comprehensive profiles of clinics or practices.
*   While official websites provided higher confidence and more complete phone numbers (where available), they only represented 8 out of 40 sources, limiting their overall impact on data completeness.
*   The reliance on directory sources for the bulk of the records contributes to the "Medium" confidence levels and the paucity of detailed contact information.

## Conclusion
This research successfully compiled a list of 32 verified neurologists and clinics in Minnesota. However, the current dataset is critically deficient in key contact information such as addresses and email addresses, severely limiting its immediate practical application for direct engagement or in-depth market analysis. To enhance the value and usability of this data, future research efforts should focus on enriching the dataset with complete and accurate contact details for the identified businesses.

In [42]:
results = run_agent("Cardiologists in Birmingham")
display(results["final_output_df"])
from IPython.display import Markdown, display
display(Markdown(results["research_summary"]))

0: Uabstvincents
1: Alluscardiologists
2: Birminghamheart
3: Baptisthealthal
4: Healthline
5: Top 60 Cardiologists near Birmingham, AL


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Uabstvincents,"127 Gardendale, AL 35071","[35205 205-939-7100, 35244 205-939-7100, 35071...",[],https://uabstvincents.org/locations/cardiology...,1,High,Official Website,https://uabstvincents.org/locations/cardiology...
1,Alluscardiologists,None,[],[contact@alluscardiologists.com],https://www.alluscardiologists.com/cardiologis...,1,High,Official Website,https://www.alluscardiologists.com/cardiologis...
2,Birminghamheart,None,[],[],https://birminghamheart.com/,1,High,Official Website,https://birminghamheart.com/
3,Baptisthealthal,None,"[(205) 786-2776, (210) 510-5000, 100 150 200, ...",[],https://www.baptisthealthal.com/cva,1,High,Official Website,https://www.baptisthealthal.com/cva
4,Healthline,None,"[51-60 (37) 61-70 (33, 40 (34) 41-50 (40)]",[],https://care.healthline.com/find-care/specialt...,1,High,Official Website,https://care.healthline.com/find-care/specialt...
5,"Top 60 Cardiologists near Birmingham, AL",None,[],[],https://www.vitals.com/cardiovascular-disease/...,1,Medium,Directory,https://www.vitals.com/cardiovascular-disease/...


# Business Research Report

## Search Summary
-- Query : Cardiologists in Birmingham
-- Businesses Found : 13
-- Businesses Verified : 6
-- Duplicate Records Removed: 7
-- Sources Searched : 40

## Executive Summary
This report details the research findings for cardiologists in Birmingham. Out of 13 businesses initially identified, 6 were successfully verified, with a significant number of duplicate records removed. Data completeness is a key challenge, particularly for addresses and email contacts, which are available for only 17% of records. Phone numbers are more frequently available (50%). The research leveraged a diverse set of sources, with official websites and directories being primary contributors, indicating a need for comprehensive data consolidation to build complete profiles.

## Key Business Identified
-- Business Name: Uabstvincents
-- Website: https://uabstvincents.org/locations/cardiology-specialists-of-birmingham/
-- Source Count: 1
-- Confidence Level: High
-- Address: 127 Gardendale, AL 35071
-- Phone Numbers: 35205 205-939-7100, 35244 205-939-7100, 35071 205-939-7100, 35242 205-939-7100

-- Business Name: Alluscardiologists
-- Website: https://www.alluscardiologists.com/cardiologists/alabama/birmingham
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Birminghamheart
-- Website: https://birminghamheart.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Baptisthealthal
-- Website: https://www.baptisthealthal.com/cva
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: (205) 786-2776, (210) 510-5000, 100 150 200, (205) 510-5000, 5 10 20 30 40 50 75

-- Business Name: Healthline
-- Website: https://care.healthline.com/find-care/specialty/cardiology/al/birmingham
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: 51-60 (37) 61-70 (33, 40 (34) 41-50 (40)

-- Business Name: Top 60 Cardiologists near Birmingham, AL
-- Website: https://www.vitals.com/cardiovascular-disease/al/birmingham
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    -- Directory: 22
    -- Official Website: 14
    -- Social Media: 4
-- Observations about source reliability: The research utilized a mix of source types. Directories were the most frequent source for information, followed by official websites. Official websites generally provided high confidence records, indicating their reliability for specific data points, though directory sources also contributed valuable information.

## Data Quality Summary
-- Records with Phone Number : 50%
-- Records with Address : 17%
-- Records with Email : 17%

## Important Findings
*   A significant number of initial records (7 out of 13) were identified as duplicates and subsequently removed.
*   Only 6 out of the 13 initially found businesses were verified, indicating challenges in confirming distinct entities.
*   Data completeness for addresses and email contacts is notably low, with only 17% of records containing this information.
*   Phone number availability is higher at 50%, although some phone number formats appear to include irrelevant numerical sequences.
*   Official websites were key contributors for high-confidence data, while directories represented the most frequently used source type.

## Conclusion
The research successfully identified and verified 6 cardiologist-related businesses in Birmingham, utilizing 40 diverse sources. While a solid foundation of business names and websites has been established, there is a clear need for further data enrichment, particularly concerning complete and accurate contact information like addresses and emails. The high confidence levels for the verified records suggest that existing data from official websites is reliable, but its scarcity highlights an area for future data collection efforts.

In [45]:
results = run_agent("Electricians in Chennai")
display(results["final_output_df"])
display(Markdown(results["research_summary"]))

0: Justdial
1: Nobroker
2: Electriciansindia
3: Datagemba
4: Urbancompany
5: Electricianchennai
6: Indiamart


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Justdial,None,[],[],https://www.justdial.com/Chennai/Electricians/...,1,High,Official Website,https://www.justdial.com/Chennai/Electricians/...
1,Nobroker,None,"[5 107334 4 2490 3 8, 60 2 794 1 3660 0]",[],https://www.nobroker.in/electrician-services-i...,1,High,Official Website,https://www.nobroker.in/electrician-services-i...
2,Electriciansindia,None,[+91 73562 32735],[support@electriciansindia.com],https://electriciansindia.com/chennai,1,High,Official Website,https://electriciansindia.com/chennai
3,Datagemba,None,[],[],https://datagemba.com/b/v/in/chennai/electricians,1,High,Official Website,https://datagemba.com/b/v/in/chennai/electricians
4,Urbancompany,5 Excellent workmanship Sathya February 2026,[],[],https://www.urbancompany.com/chennai-electricians,1,High,Official Website,https://www.urbancompany.com/chennai-electricians
5,Electricianchennai,None,"[+91 9597777186, +91 9677761591]",[electricianinchennai7@gmail.com],https://electricianchennai.com/,1,High,Official Website,https://electricianchennai.com/
6,Indiamart,None,[],[],https://dir.indiamart.com/chennai/electrical-w...,1,High,Official Website,https://dir.indiamart.com/chennai/electrical-w...


# Business Research Report

## Search Summary
-- Query : Electricians in Chennai
-- Businesses Found : 16
-- Businesses Verified : 7
-- Duplicate Records Removed: 9
-- Sources Searched : 40

## Executive Summary
This research aimed to identify electricians in Chennai, leveraging 40 diverse sources. Out of 16 businesses identified, 7 were verified, with 9 duplicate records removed. The findings indicate that the most prominent entities found are online directories and service aggregation platforms rather than individual electrician businesses. Data quality for direct contact information such as physical addresses (14%), phone numbers (43%), and emails (29%) is notably low, suggesting challenges in obtaining comprehensive contact details for service providers in this market.

## Key Business Identified

-- **Business Name**: Justdial
-- **Website**: https://www.justdial.com/Chennai/Electricians/nct-10184166
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: Not Available

-- **Business Name**: Nobroker
-- **Website**: https://www.nobroker.in/electrician-services-in-chennai
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: 5 107334 4 2490 3 8, 60 2 794 1 3660 0

-- **Business Name**: Electriciansindia
-- **Website**: https://electriciansindia.com/chennai
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: +91 73562 32735

-- **Business Name**: Datagemba
-- **Website**: https://datagemba.com/b/v/in/chennai/electricians
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: Not Available

-- **Business Name**: Urbancompany
-- **Website**: https://www.urbancompany.com/chennai-electricians
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: 5 Excellent workmanship Sathya February 2026
-- **Phone Numbers**: Not Available

-- **Business Name**: Electricianchennai
-- **Website**: https://electricianchennai.com/
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: +91 9597777186, +91 9677761591

-- **Business Name**: Indiamart
-- **Website**: https://dir.indiamart.com/chennai/electrical-work.html
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: Not Available
-- **Phone Numbers**: Not Available

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    -- Official Website: 19
    -- Search Engine: 16
    -- Social Media: 4
    -- Directory: 1
-- Observations about source reliability: Official Websites and Search Engines were the most utilized sources, suggesting a strong digital presence for information related to electricians in Chennai. The consistent "High" confidence level across the verified records indicates that the information, when found, was reliably sourced, often directly from the entity's declared online presence.

## Data Quality Summary
-- Records with Phone Number : 43%
-- Records with Address : 14%
-- Records with Email : 29%

## Important Findings
*   A significant challenge was the presence of duplicate records, with 9 identified and removed, indicating potential inefficiencies or overlaps across various data sources.
*   Only 7 out of 16 businesses were successfully verified, resulting in a verification rate of approximately 44%, despite an extensive search across 40 sources.
*   The majority of identified entities are online platforms and directories (e.g., Justdial, Nobroker, Urbancompany) rather than direct individual electrician service providers.
*   There is a notable lack of comprehensive contact information; specifically, addresses were available for only 14% of records, phone numbers for 43%, and emails for 29%.
*   "Official Website" was the most frequently tapped source type, yet it often led to aggregator sites, suggesting that individual electricians may have a less direct or easily discoverable online footprint.

## Conclusion
This business research provided an overview of entities related to electricians in Chennai, predominantly identifying digital platforms and directories. While a broad search was conducted across 40 sources, the relatively low business verification rate and the high number of duplicate records suggest complexities in data aggregation for this query. More critically, the poor data quality for direct contact details (address, phone, email) indicates a significant challenge in compiling actionable lists of individual electrician services. Future research may benefit from refining search strategies to target individual service providers more effectively, possibly by leveraging the aggregator sites found, or by exploring local offline directories and recommendations.

In [46]:
results = run_agent("Dentists in Lakeway Texas")
display(results["final_output_df"])
display(Markdown(results["research_summary"]))

0: Familydentistlakeway
1: Lakeway Dental
2: Lakewaydentalassociates
3: Patientconnect365
4: Denscore
5: Zocdoc
6: Dental
7: Opencare
8: Superpages
9: Dentistdig
10: LAKEWAY COSMETIC DENTISTRY


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Familydentistlakeway,"1913 Ranch Road 620 S Ste 101 Lakeway, TX 7873...","[(512) 788-9004, (512) 788-9001, 2020-2023]",[scheduling@familydentistlakeway.com],https://familydentistlakeway.com/,1,High,Official Website,https://familydentistlakeway.com/
1,Lakeway Dental,"00 2220 Lakeway Blvd, Lakeway TX 78734",[(512) 261-5522],[info@lakeway-dental.com],https://www.lakeway-dental.com/,1,High,Official Website,https://www.lakeway-dental.com/
2,Lakewaydentalassociates,None,[],[],https://www.lakewaydentalassociates.com/,1,High,Official Website,https://www.lakewaydentalassociates.com/
3,Patientconnect365,"6 , Lakeway, TX 78734","[512-263-0064, - 78734 - 512-263-0]",[],https://patientconnect365.com/Dentists/Texas/L...,1,High,Official Website,https://patientconnect365.com/Dentists/Texas/L...
4,Denscore,"250, Lakeway, TX 78738","[(737) 238-6528, (512) 402-9090, (512) 368-616...",[],https://www.denscore.com/the-30-best-dentists-...,1,High,Official Website,https://www.denscore.com/the-30-best-dentists-...
5,Zocdoc,None,[],[],https://www.zocdoc.com/dentists/lakeway-221949pm,1,High,Official Website,https://www.zocdoc.com/dentists/lakeway-221949pm
6,Dental,None,[],[],https://dental.me/blog/finding-a-dentist-in-la...,1,High,Official Website,https://dental.me/blog/finding-a-dentist-in-la...
7,Opencare,15401 Nightingale Lane Lakeway TX 78734,"[78613 (180), 78746 (86), 78749 (20), 78734 (12)]",[],https://www.opencare.com/dentists/lakeway-tx/,1,High,Official Website,https://www.opencare.com/dentists/lakeway-tx/
8,Superpages,0064 Call Now Visit Website Directions Dentist...,"[512-263-8989, 512-745-4391, 78734 14, 78734 1...",[],https://www.superpages.com/lakeway-tx/dentists,1,High,Official Website,https://www.superpages.com/lakeway-tx/dentists
9,Dentistdig,None,[],[],https://dentistdig.com/dentists/tx/lakeway,1,High,Official Website,https://dentistdig.com/dentists/tx/lakeway


# Business Research Report

## Search Summary
-- Query : Dentists in Lakeway Texas
-- Businesses Found : 22
-- Businesses Verified : 11
-- Duplicate Records Removed: 11
-- Sources Searched : 40

## Executive Summary
This report details the findings from a business research query for "Dentists in Lakeway Texas". Out of 22 businesses found, 11 unique records were verified after removing duplicates. While a significant number of sources (40) were consulted, the overall data quality for key contact information such as phone numbers, addresses, and especially emails, remains moderate to low, with only 18% of records containing email addresses. "Official Website" was the most common and reliable source type.

## Key Business Identified

-- Business Name: Familydentistlakeway
-- Website: https://familydentistlakeway.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: 1913 Ranch Road 620 S Ste 101 Lakeway, TX 78734 Make Appointment Home About Us Our Services Cosmetic Dentistry Restorative Services Surgical Services Sedation Dentistry Prevention Dentistry State of the Art Technology New Patients Promotions Blog Gallery Contact Contact Info Address 1913 Ranch Road 620 S Ste 101 Lakeway, TX 78734
-- Phone Numbers: (512) 788-9004, (512) 788-9001, 2020-2023

-- Business Name: Lakeway Dental
-- Website: https://www.lakeway-dental.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: 00 2220 Lakeway Blvd, Lakeway TX 78734
-- Phone Numbers: (512) 261-5522

-- Business Name: Lakewaydentalassociates
-- Website: https://www.lakewaydentalassociates.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Patientconnect365
-- Website: https://patientconnect365.com/Dentists/Texas/Lakeway/78734/Lakeway_Dental_Associates
-- Source Count: 1
-- Confidence Level: High
-- Address: 6 , Lakeway, TX 78734
-- Phone Numbers: 512-263-0064, - 78734 - 512-263-0

-- Business Name: Denscore
-- Website: https://www.denscore.com/the-30-best-dentists-in-lakeway-tx/
-- Source Count: 1
-- Confidence Level: High
-- Address: 250, Lakeway, TX 78738
-- Phone Numbers: (737) 238-6528, (512) 402-9090, (512) 368-6165, (512) 334-0345, (512) 271-6600, (512) 266-9585, (512) 717-4786, (512) 263-5566, (512) 382-6985, (512) 710-4783, (512) 266-1339

-- Business Name: Zocdoc
-- Website: https://www.zocdoc.com/dentists/lakeway-221949pm
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dental
-- Website: https://dental.me/blog/finding-a-dentist-in-lakeway-tx/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Opencare
-- Website: https://www.opencare.com/dentists/lakeway-tx/
-- Source Count: 1
-- Confidence Level: High
-- Address: 15401 Nightingale Lane Lakeway TX 78734
-- Phone Numbers: 78613 (180), 78746 (86), 78749 (20), 78734 (12)

-- Business Name: Superpages
-- Website: https://www.superpages.com/lakeway-tx/dentists
-- Source Count: 1
-- Confidence Level: High
-- Address: 0064 Call Now Visit Website Directions Dentists 1310
-- Phone Numbers: 512-263-8989, 512-745-4391, 78734 14, 78734 17, 4807 183, 1526 1 2 3, 512-219-1747, 78602 737-278-7524, 78756 737-381-1536, 78734 22, 512-520-9911, 78734 18, 512-494-6958, 78734 25, 78734 15, 78734 20, 512-263-8284, 512-619-3514, 512-263-1661, 78613 512-813-1191, 78704 737-381-3083, 78753 512-501-1890, 800-761-7166, 512-716-0307, 78734 23, 78734 13, 512-587-7979, 512-261-5522, 78734 10, 512-263-8337, 512-261-4590, 512-263-4252, 512-263-9544, 78626 512-887-3813, 512-402-9399, 78734 19, 78734 16, 512-334-0345, 78734 21, 512-263-0064, 512-368-6165, 512-705-6507, 78734 12, 78738 28, 78640 830-310-7315, 512-660-6006, 512-769-7964

-- Business Name: Dentistdig
-- Website: https://dentistdig.com/dentists/tx/lakeway
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: LAKEWAY COSMETIC DENTISTRY
-- Website: https://www.yelp.ca/biz/lakeway-cosmetic-dentistry-lakeway-2
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    -- Official Website: 23
    -- Directory: 12
    -- Search Engine: 3
    -- Social Media: 2
-- Observations about source reliability: "Official Website" is the most prevalent source type, contributing to a high confidence level for most identified businesses. Directory sources also play a significant role. The confidence levels generally reflect the expected reliability of these source types.

## Data Quality Summary
-- Records with Phone Number : 55%
-- Records with Address : 55%
-- Records with Email : 18%

## Important Findings
*   A substantial number of initial business records (22) were found, indicating a competitive market for dentists in Lakeway, Texas.
*   Half of the initially found records (11 out of 22) were identified as duplicates, highlighting the need for robust de-duplication processes in data collection.
*   While 11 unique businesses were verified with high confidence, the availability of comprehensive contact information is limited, particularly for email addresses (only 18% of records).
*   "Official Website" is the primary source of information, suggesting direct verification is a strong factor in data confidence for these records.
*   Several identified "businesses" such as Zocdoc, Denscore, Superpages, Opencare, and Patientconnect365 appear to be directories or aggregators rather than individual dental practices, which could skew the count of actual dental offices.

## Conclusion
The research successfully identified 11 verified businesses related to dentists in Lakeway, Texas, with generally high confidence in the provided information. However, the data quality for direct contact methods, especially email, is low, which could impact direct outreach efforts. Future research might benefit from deeper dives into the websites of directory-type listings to extract individual dental practice details.